# 0.3 RPC BAL RLP Estimation

This notebook estimates raw RLP BAL bytes using JSON-RPC `debug_traceBlockByNumber` with `prestateTracer`, following the `eth-bal-analysis` builder logic.

It does **not** compute calldata from RPC. It reads calldata bytes from the CSV produced by `0.2-calldata-xatu.ipynb`, writes a separate RPC BAL summary CSV, and merges BAL bytes back into the calldata CSV.

## Bandwidth Join

```text
bandwidth_rlp_bytes = xatu_calldata_bytes + rpc_bal_rlp_bytes
```

Each BAL account entry is encoded as:

```text
[address, storage_writes, storage_reads, balance_changes, nonce_changes, code_changes]
```

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim.rpc_bal import build_rpc_bal_for_block

load_dotenv(PROJECT_ROOT / ".env")
RPC_URL = os.environ.get("ALCHEMY_RPC")
if not RPC_URL:
    raise RuntimeError("Missing ALCHEMY_RPC in .env")

# Do not print RPC_URL; it contains the API key.
print("Loaded ALCHEMY_RPC")

In [ ]:
START_BLOCK = 22_886_891
N_BLOCKS = 50
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
CALLDATA_CSV = PROJECT_ROOT / "data" / f"xatu_calldata_{min(BLOCKS)}_{max(BLOCKS)}.csv"
SUMMARY_CSV = PROJECT_ROOT / "data" / f"rpc_bal_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"

INCLUDE_READS = True

# True matches the current eth-bal-analysis builder path.
INCLUDE_SYSTEM_CHANGES = True

WRITE_CSV = True
WRITE_RLP = False

In [ ]:
if not CALLDATA_CSV.exists():
    raise FileNotFoundError(
        f"Missing calldata CSV: {CALLDATA_CSV}. Run notebooks/0.2-calldata-xatu.ipynb first."
    )

calldata = pd.read_csv(CALLDATA_CSV)
missing_blocks = sorted(set(BLOCKS) - set(calldata["block_number"].astype(int)))
if missing_blocks:
    raise RuntimeError(f"Calldata CSV is missing blocks: {missing_blocks}")

calldata_by_block = calldata.set_index("block_number")["calldata_bytes"].astype(int).to_dict()
display(calldata)

In [ ]:
summary_cols = [
    "block_number",
    "include_reads",
    "include_system_changes",
    "calldata_source",
    "calldata_bytes",
    "bal_rlp_bytes",
    "bandwidth_rlp_bytes",
    "accounts",
    "storage_write_slots",
    "storage_write_changes",
    "storage_reads",
    "balance_changes",
    "nonce_changes",
    "code_changes",
    "code_bytes",
    "storage_writes_rlp_bytes",
    "storage_reads_rlp_bytes",
    "balance_changes_rlp_bytes",
    "nonce_changes_rlp_bytes",
    "code_changes_rlp_bytes",
    "account_shell_rlp_bytes",
]

rows = []
if SUMMARY_CSV.exists():
    prior = pd.read_csv(SUMMARY_CSV)
    if "include_reads" in prior.columns and "include_system_changes" in prior.columns:
        prior = prior[
            (prior["include_reads"].astype(bool) == INCLUDE_READS)
            & (prior["include_system_changes"].astype(bool) == INCLUDE_SYSTEM_CHANGES)
        ]
    else:
        prior = prior.iloc[0:0]
    available_summary_cols = [col for col in summary_cols if col in prior.columns]
    rows.extend(prior[available_summary_cols].to_dict("records"))

seen = {int(row["block_number"]) for row in rows}
rlp_outputs = {}

for block_number in BLOCKS:
    if int(block_number) in seen:
        print(f"Skipping block {block_number}; already in {SUMMARY_CSV.name}")
        continue
    print(f"Building RPC BAL for block {block_number}...")
    result = build_rpc_bal_for_block(
        RPC_URL,
        block_number,
        calldata_bytes=calldata_by_block[block_number],
        include_reads=INCLUDE_READS,
        include_system_changes=INCLUDE_SYSTEM_CHANGES,
    )
    rows.append(result.summary.as_dict())
    seen.add(int(block_number))
    rlp_outputs[block_number] = result.rlp_bytes
    if WRITE_CSV:
        data_dir = PROJECT_ROOT / "data"
        data_dir.mkdir(exist_ok=True)
        pd.DataFrame(rows).reindex(columns=summary_cols).drop_duplicates("block_number", keep="last").sort_values("block_number").to_csv(SUMMARY_CSV, index=False)

summary = pd.DataFrame(rows).reindex(columns=summary_cols).drop_duplicates("block_number", keep="last").sort_values("block_number")
display(summary)

if WRITE_CSV:
    data_dir = PROJECT_ROOT / "data"
    data_dir.mkdir(exist_ok=True)
    summary.to_csv(SUMMARY_CSV, index=False)
    print(SUMMARY_CSV)

merge_cols = [col for col in summary_cols if col not in {"calldata_bytes", "calldata_source"}]
stale_cols = [col for col in merge_cols if col != "block_number" and col in calldata.columns]
merged = calldata.drop(columns=stale_cols + [col for col in ["bal_bytes", "bandwidth_bytes"] if col in calldata.columns])
merged = merged.merge(summary[merge_cols], on="block_number", how="left", validate="one_to_one")
merged["bal_bytes"] = merged["bal_rlp_bytes"].astype("Int64")
merged["bandwidth_bytes"] = (merged["calldata_bytes"] + merged["bal_bytes"]).astype("Int64")

front = []
for col in merged.columns:
    if col in {"bal_bytes", "bandwidth_bytes"}:
        continue
    front.append(col)
    if col == "calldata_bytes":
        front.extend(["bal_bytes", "bandwidth_bytes"])
merged = merged[front + [col for col in merged.columns if col not in front]]
display(merged)

if WRITE_CSV:
    merged.to_csv(CALLDATA_CSV, index=False)
    print(CALLDATA_CSV)

if WRITE_RLP:
    suffix = "with_reads" if INCLUDE_READS else "without_reads"
    for block_number, payload in rlp_outputs.items():
        data_dir = PROJECT_ROOT / "data"
        data_dir.mkdir(exist_ok=True)
        out = data_dir / f"rpc_bal_{block_number}_{suffix}.rlp"
        out.write_bytes(payload)
        print(out)

In [ ]:
# Optional local calibration against nerolation/eth-bal-analysis raw RLP samples.
sample_dir = Path("/private/tmp/eth-bal-analysis/bal_raw/rlp")
calibration_rows = []
if sample_dir.exists():
    suffix = "with_reads" if INCLUDE_READS else "without_reads"
    for block_number in BLOCKS:
        sample = sample_dir / f"{block_number}_{suffix}.rlp"
        if sample.exists():
            sample_bytes = sample.stat().st_size
            row = summary[summary["block_number"] == block_number].iloc[0]
            calibration_rows.append({
                "block_number": block_number,
                "rpc_bal_rlp_bytes": int(row["bal_rlp_bytes"]),
                "sample_bal_rlp_bytes": sample_bytes,
                "delta_bytes": int(row["bal_rlp_bytes"]) - sample_bytes,
            })

calibration = pd.DataFrame(calibration_rows)
display(calibration)